TV, radyo ve gazeteye ayrı ayrı ne kadar yatırım yaparsam satış rakamlarım ne kadar artar sorusuna cevap arayacaksınız ve  
Hangi reklam yönteminin daha başarılı olduğunu bulacaksınız.

In [2]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

In [3]:
df = pd.read_csv("advertising.csv")

In [4]:
df.head()

,TV,Radio,Newspaper,Sales
0,230.1,37.8,69.2,22.1
1,44.5,39.3,45.1,10.4
2,17.2,45.9,69.3,9.3
3,151.5,41.3,58.5,18.5
4,180.8,10.8,58.4,12.9


In [11]:
df.corr()

,TV,Radio,Newspaper,Sales
TV,1.000000,0.054809,0.056648,0.782224
Radio,0.054809,1.000000,0.354104,0.576223
Newspaper,0.056648,0.354104,1.000000,0.228299
Sales,0.782224,0.576223,0.228299,1.000000


In [19]:
abs(df.corr(numeric_only=True)['Sales'].sort_values(ascending=False)) # Price ile en cok korelasyon olan sutunlar

Sales        1.000000
TV           0.782224
Radio        0.576223
Newspaper    0.228299
Name: Sales, dtype: float64

In [26]:
x = df.drop("Sales", axis=1)
y = df["Sales"]

In [27]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns",100)

from sklearn.linear_model import LinearRegression,SGDRegressor,Ridge,Lasso,ElasticNet
from sklearn.neighbors import KNeighborsRegressor, RadiusNeighborsRegressor
from sklearn.ensemble import GradientBoostingRegressor,AdaBoostRegressor
from sklearn.tree import DecisionTreeRegressor, plot_tree, ExtraTreeRegressor
#pip install xgboost
from xgboost import XGBRegressor
from sklearn.svm import SVR

from sklearn.neural_network import MLPRegressor

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error,r2_score,mean_absolute_error

from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler

def algo_test(x,y):
        #Bütün modelleri tanımlıyorum
        L=LinearRegression()
        R=Ridge()
        Lass=Lasso()
        E=ElasticNet()
        sgd=SGDRegressor()
        ETR=ExtraTreeRegressor()
        GBR=GradientBoostingRegressor()
        kn=KNeighborsRegressor()
        rkn=RadiusNeighborsRegressor(radius=1.0)
        ada=AdaBoostRegressor()
        dt=DecisionTreeRegressor()
        xgb=XGBRegressor()
        svr=SVR()
        mlp_regressor = MLPRegressor()

       
        
        algos=[L,R,Lass,E,sgd,ETR,GBR,ada,kn,dt,xgb,svr,mlp_regressor]
        algo_names=['Linear','Ridge','Lasso','ElasticNet','SGD','Extra Tree','Gradient Boosting',
                    'KNeighborsRegressor','AdaBoost','Decision Tree','XGBRegressor','SVR','mlp_regressor']
        x=MinMaxScaler().fit_transform(x)
        x_train, x_test, y_train, y_test=train_test_split(x,y,test_size=.20,random_state=42)
        
        r_squared= []
        rmse= []
        mae= []
        
        #Hata ve doğruluk oranlarını bir tablo haline getirmek için bir dataframe oluşturuyorum
        result=pd.DataFrame(columns=['R_Squared','RMSE','MAE'],index=algo_names)
        
        
        for algo in algos:
            p=algo.fit(x_train,y_train).predict(x_test)
            r_squared.append(r2_score(y_test,p))
            rmse.append(mean_squared_error(y_test,p)**.5)
            mae.append(mean_absolute_error(y_test,p))
        
            

        #result adlı tabloya doğruluk ve hata oranlarımı yerleştiriyorum
        result.R_Squared=r_squared
        result.RMSE=rmse
        result.MAE=mae
        
       #oluşturduğum result tablosunu doğruluk oranına (r2_score) göre sıralayıp dönüyor
        rtable=result.sort_values('R_Squared',ascending=False)
        return rtable

In [28]:
algo_test(x,y)

,R_Squared,RMSE,MAE
Gradient Boosting,0.983536,0.720869,0.611934
XGBRegressor,0.972470,0.932166,0.725829
Extra Tree,0.968310,1.000125,0.782500
KNeighborsRegressor,0.962710,1.084898,0.895437
Decision Tree,0.950441,1.250700,0.902500
SVR,0.934851,1.433997,0.968690
AdaBoost,0.933719,1.446396,0.980000
Linear,0.899438,1.781600,1.460757
Ridge,0.889218,1.869940,1.525709
SGD,0.884962,1.905521,1.540769


In [31]:
lin = LinearRegression()

In [29]:
x=MinMaxScaler().fit_transform(x)
x_train, x_test, y_train, y_test=train_test_split(x,y,test_size=.20,random_state=42)

In [32]:
model = lin.fit(x_train,y_train)
tahmin = model.predict(x_test)

In [33]:
r2_score(y_test,tahmin)

0.899438024100912

In [35]:
coef = model.coef_

In [36]:
coef

array([13.22651832,  9.38407469,  0.3139387 ])

In [38]:
print("\nKanal etkileri (katsayı):")

for col, value in zip(['TV', 'Radio', 'Newspaper'], coef):
    print(col, "=>", value)


Kanal etkileri (katsayı):
TV => 13.226518315499417
Radio => 9.384074690025068
Newspaper => 0.31393870061344753


#### Tv, Radio, Newspaperin sales'e olan katsayisini ve p degerlerini bulabilmek icin statsmodels kullanilmasi lazim.    
#### ondan dolayi OLS modeliyle egitip katsayilarini ve p degerleri bulup hangi reklam yonteminin daha basarili oldugunu bulacagiz.

In [40]:
import statsmodels.api as sm

X = df[['TV', 'Radio', 'Newspaper']]
y = df['Sales']

# sabit terim ekle
X = sm.add_constant(X)

model = sm.OLS(y, X).fit()

print(model.summary())

                            OLS Regression Results                            
Dep. Variable:                  Sales   R-squared:                       0.897
Model:                            OLS   Adj. R-squared:                  0.896
Method:                 Least Squares   F-statistic:                     570.3
Date:                Mon, 25 May 2026   Prob (F-statistic):           1.58e-96
Time:                        14:52:01   Log-Likelihood:                -386.18
No. Observations:                 200   AIC:                             780.4
Df Residuals:                     196   BIC:                             793.6
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          2.9389      0.312      9.422      0.0

In [41]:
coef = model.params 

In [43]:
print("\nSales'e olan Kanal etkileri (katsayı):")
for col in ['TV', 'Radio', 'Newspaper']:
    print(col, "=>", coef[col])


Sales'e olan Kanal etkileri (katsayı):
TV => 0.04576464545539764
Radio => 0.1885300169182044
Newspaper => -0.0010374930424762313


In [44]:
pvals = model.pvalues
print("\nP-değerleri:")
for col in ['TV', 'Radio', 'Newspaper']:
    print(col, "=>", pvals[col])


P-değerleri:
TV => 1.5099599548142635e-81
Radio => 1.5053389205756157e-54
Newspaper => 0.8599150500805794


1. p < 0.05 ise: etki daha güvenilir yorumlanır.  
2. Sonra aralarında en büyük (mutlak ya da pozitif) katsayı olan “en etkili” diye seçilir.

#### Yukarida baktigimiz linear regression katsayilarini standartlastirp kiyaslama yapip neticeyi gorelim.

In [49]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df[['TV', 'Radio', 'Newspaper']])

X_train = X_scaled
y_train = df['Sales'].values


lin.fit(X_train, y_train)

coefs_scaled = pd.Series(lin.coef_, index=['TV','Radio','Newspaper'])
print("Standartlaştırılmış katsayılar:")
print(coefs_scaled)

print("\nEn etkili (en büyük katsayı):")
print(coefs_scaled.sort_values(ascending=False))

Standartlaştırılmış katsayılar:
TV           3.919254
Radio        2.792063
Newspaper   -0.022539
dtype: float64

En etkili (en büyük katsayı):
TV           3.919254
Radio        2.792063
Newspaper   -0.022539
dtype: float64


In [50]:
lin.coef_

array([ 3.91925365,  2.79206274, -0.02253861])